# Workday Reports to Databricks Pipeline Generation

This notebook generates Workday Reports to Databricks ingestion pipelines using Databricks Asset Bundles.

## Process Overview
1. **Pipeline grouping**: Groups Workday reports by prefix + priority combinations
2. **YAML generation**: Creates Databricks Asset Bundle YAML files

## Key Features
- **No Gateways**: Workday Reports is a SaaS connector and does NOT require gateway configuration
- **Primary Keys Required**: Each report must specify primary key columns
- **SCD Type Support**: Choose between SCD_TYPE_1 (overwrite) or SCD_TYPE_2 (history tracking)
- **Column Filtering**: Include or exclude specific columns per report (optional)
- **Prefix + Priority Grouping**: Each unique (prefix, priority) pair becomes a separate pipeline

## Prerequisites
- Workday connection configured in Databricks
- CSV file with your Workday report URLs
- Databricks workspace with Unity Catalog enabled

In [ ]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules

In [ ]:
%pip install pyyaml

## Configure Parameters

**IMPORTANT**: Modify these parameters for your environment before running!

In [ ]:
from databricks.sdk import WorkspaceClient
from utilities import load_input_csv
from workday_reports.connector import WorkdayReportsConnector

w = WorkspaceClient()

# Automatically get current user email
username = w.current_user.me().user_name

# Automatically get workspace host URL
workspace_host = w.config.host

# Automatically construct output directory
WORKSPACE_HOST = workspace_host
ROOT_PATH = f'/Users/{username}/.bundle/${{bundle.name}}/${{bundle.target}}'

print(f"✓ User: {username}")
print(f"✓ Workspace: {WORKSPACE_HOST}")
print(f"✓ Bundle Root Path: {ROOT_PATH}")

## Generate Pipelines

Run this cell to generate the DAB project with pipelines and jobs.

In [ ]:
# Configure your paths
INPUT_CSV = 'examples/basic/pipeline_config.csv'
OUTPUT_DIR = f'/Workspace/Users/{username}/lakehouse-tapworks/workday_reports/deployment'

# Load input CSV
df = load_input_csv(INPUT_CSV)

print(f"\n📊 Loaded {len(df)} reports from {INPUT_CSV}")

# Create connector
connector = WorkdayReportsConnector()

# Generate pipelines
result = connector.run_complete_pipeline_generation(
    df=df,
    output_dir=OUTPUT_DIR,
    targets={
        'dev': {
            'workspace_host': WORKSPACE_HOST,
            'root_path': ROOT_PATH
        }
    },
    max_tables_per_pipeline=250
)

print(f"\n✅ Pipeline generation complete!")
print(f"📁 Output directory: {OUTPUT_DIR}")

In [ ]:
# Display summary
print(f"Total reports: {len(result)}")
print(f"\nPipelines created: {result['pipeline_group'].nunique()}")
print(f"\nPipeline groups:")
print(result.groupby('pipeline_group').size())

# Show first few rows
display(result[[
    'source_url', 
    'target_table_name', 
    'pipeline_group', 
    'schedule',
    'scd_type',
    'primary_keys'
]].head(10))

## Advanced: Using Custom Default Values and Overrides

You can use `default_values` and `override_input_config` parameters for advanced configuration:

**`default_values` Parameter:**
- Provides custom default values for optional columns
- Merged with built-in defaults (your values take precedence)
- Applied only to missing/empty values in the CSV

**`override_input_config` Parameter:**
- Forces specific column values for ALL rows
- Overwrites any existing values in the CSV
- Useful for environment-specific overrides

**Example Use Cases:**
- Override schedule for testing (e.g., disable auto-runs)
- Force all pipelines to use a specific catalog/schema for dev environment
- Change SCD type for all reports

In [ ]:
INPUT_CSV = 'examples/basic/pipeline_config.csv'
OUTPUT_DIR = f'/Workspace/Users/{username}/lakehouse-tapworks/workday_reports/deployment_dev'

df = load_input_csv(INPUT_CSV)

# Create connector
connector = WorkdayReportsConnector()

# Generate pipelines with overrides
result = connector.run_complete_pipeline_generation(
    df=df,
    output_dir=OUTPUT_DIR,
    targets={
        'dev': {
            'workspace_host': WORKSPACE_HOST,
            'root_path': ROOT_PATH
        }
    },
    max_tables_per_pipeline=250,
    # Override for dev environment
    override_input_config={
        'target_catalog': 'dev_catalog',
        'schedule': '0 0 * * 0'  # Weekly on Sundays for dev
    },
    # Custom defaults
    default_values={
        'scd_type': 'SCD_TYPE_2'  # Force history tracking
    }
)

## Example CSV Format

```csv
source_url,target_catalog,target_schema,target_table_name,prefix,priority,connection_name,schedule,include_columns,exclude_columns,scd_type,primary_keys
https://wd2-impl-services1.workday.com/ccx/service/customreport2/tenant/report1,main,workday,employee_data,hr,01,workday_connection,0 */6 * * *,,,SCD_TYPE_2,"Employee_ID"
https://wd2-impl-services1.workday.com/ccx/service/customreport2/tenant/report2,main,workday,time_tracking,hr,01,workday_connection,0 */6 * * *,,,SCD_TYPE_2,"Worker_ID,Effective_Date"
https://wd2-impl-services1.workday.com/ccx/service/customreport2/tenant/report3,main,workday,cost_centers,finance,01,workday_connection,0 0 * * *,,,SCD_TYPE_1,"Cost_Center_ID"
```

**Important Notes:**
- **Primary Keys Required**: Unlike ServiceNow, Workday Reports REQUIRE primary key columns to be specified
- Each report must have at least one primary key
- Multiple primary keys can be specified as comma-separated values (e.g., "Worker_ID,Effective_Date")

## Next Steps

After generating the DAB project:

1. **Review Generated Files**:
   - Navigate to the output directory
   - Review `databricks.yml`, `resources/pipelines.yml`, and `resources/jobs.yml`

2. **Deploy Using Databricks CLI**:
   ```bash
   cd {OUTPUT_DIR}
   databricks bundle validate -t dev
   databricks bundle deploy -t dev
   ```

3. **Monitor Pipelines**:
   - Go to Databricks workspace → Workflows
   - Find jobs named "Workday Reports Pipeline Scheduler - ..."
   - Check pipeline runs and status

4. **Query Your Data**:

## Troubleshooting

**Connection Issues:**
- Verify Workday connection is active in Databricks UI (Catalog → Connections)
- Check connection name in CSV matches the actual connection name
- Ensure credentials are correctly configured

**Pipeline Failures:**
- Verify report URLs are accessible and correct
- Check primary_keys column names match actual report schema
- Check include/exclude columns match actual report schema
- Review pipeline logs in Databricks UI

**Primary Key Errors:**
- Each report MUST have at least one primary key specified
- Primary key column names must match exactly (case-sensitive)
- For composite keys, use comma-separated format: "Column1,Column2"

**Performance:**
- Adjust schedules based on report update frequency
- Use column filtering to reduce data volume
- Group reports by update frequency using priority levels